In [ ]:
# Step 1: match_sop(Diagnose incident window 2024-01-19 12:40:21 to 2024-01-19 13:10:21 UTC. Telemetry available: metric.csv, trace.csv, log
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.55)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.47)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Log-Based Reason Analysis SOP (score 0.46)
  Name: RCA-Agent Log-Based Reason Analysis SOP
Steps:
1. get_logs(None, start_time, end_time): summarize log anomalies and raw error messages for candidate components in the incident window.
2. get_relevant_metric('error'): list error, resource, and traffic-related metric fields that may explain the logs.
3. whether_is_abnormal_metric(start_time, end_time, '', None): compare log evidence against metric anomalies and onset times.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 2: generate_sop_code for 'RCA-Agent Metric Fault Identification SOP'
start_time = '2024-01-19 12:40:21'
end_time = '2024-01-19 13:10:21'
metrics = whether_is_abnormal_metric(start_time, end_time, '', None)
analysis = component_analyze(start_time, end_time)
answer = metrics + '\n' + analysis

```
Generated code for SOP 'RCA-Agent Metric Fault Identification SOP':
start_time = '2024-01-19 12:40:21'
end_time = '2024-01-19 13:10:21'
metrics = whether_is_abnormal_metric(start_time, end_time, '', None)
analysis = component_analyze(start_time, end_time)
answer = metrics + '\n' + analysis
```

In [ ]:
# Step 3: run_sop
start_time = '2024-01-19 12:40:21'
end_time = '2024-01-19 13:10:21'
metrics = whether_is_abnormal_metric(start_time, end_time, '', None)
analysis = component_analyze(start_time, end_time)
answer = metrics + '\n' + analysis

```
[ANOMALY] currencyservice socket: rise to 3.267 at 2024-01-19 12:50:00 (median 3, thr 3.133, sev 44.1, n=2)
[ANOMALY] emailservice latency-50: rise to 0.08032 at 2024-01-19 12:56:00 (median 0.003112, thr 0.0773, sev 28.2, n=2)
[ANOMALY] checkoutservice latency-50: rise to 2.866 at 2024-01-19 13:01:00 (median 0.1363, thr 2.771, sev 22.5, n=2)
[ANOMALY] checkoutservice latency-90: rise to 4.694 at 2024-01-19 13:00:00 (median 0.6449, thr 4.569, sev 6.4, n=2)
[ANOMALY] emailservice latency-90: rise to 0.2205 at 2024-01-19 13:00:00 (median 0.02595, thr 0.2202, sev 6.1, n=2)
[ANOMALY] recommendationservice latency-50: rise to 0.007545 at 2024-01-19 12:53:00 (median 0.007482, thr 0.007537, sev 4.8, n=2)
[ANOMALY] productcatalogservice mem: rise to 1.087e+07 at 2024-01-19 12:43:00 (median 1.035e+07, thr 1.07e+07, sev 4.5, n=1)
[ANOMALY] recommendationservice latency-90: rise to 0.009638 at 2024-01-19 12:49:00 (median 0.009549, thr 0.009633, sev 4.4, n=2)
[ANOMALY] currencyservice cpu: drop to 29.26 at 2024-01-19 12:47:00 (median 32.59, thr 29.66, sev 3.5, n=2)
Component anomaly overview (ranked):
recommendationservice  metric_anoms=2 (top: latency-50 sev=4.8)  trace_anoms=8  log_anoms=1
checkoutservice  metric_anoms=2 (top: latency-50 sev=22.5)  trace_anoms=8  log_anoms=0
currencyservice  metric_anoms=2 (top: socket sev=44.1)  trace_anoms=5  log_anoms=1
productcatalogservice  metric_anoms=1 (top: mem sev=4.5)  trace_anoms=6  log_anoms=0
emailservice  metric_anoms=2 (top: latency-50 sev=28.2)  trace_anoms=2  log_anoms=0
```

In [ ]:
# Step 4: match_observation([ANOMALY] currencyservice socket: rise to 3.267 at 2024-01-19 12:50:00 (median 3, thr 3.133, sev 44.1, n=2)
[ANOMALY] em
...[truncated])

```
ObAgent: The fault class is likely: socket error.

Reason: a large spike in the currencyservice "socket" metric (sev 44.1) coincides with P50/P90 latency rises across multiple services (checkout, email, recommendation), while CPU and memory show no matching systemic resource spike. This pattern fits connection/socket failures causing upstream RPC delays. Packet-loss or retransmit faults would show TCP retransmits/throughput drops (not reported); disk I/O/CPU/memory rises are not present. To confirm, inspect socket error counts, connection resets, and TCP retransmits on the affected hosts (whether_is_abnormal_metric for socket_errors and tcp_retransmits, and collect_trace for the RPCs).
JudgeAgent: Yes - Located component: currencyservice. Anomaly type: socket error. Evidence: metric anomaly 'currencyservice socket' rose to 3.267 at 2024-01-19 12:50:00 (median 3, threshold 3.133, severity 44.1), matching upstream P50/P90 latency increases—concrete telemetry pins the socket error as root cause.
```